# PanAf Ape Detection — Phase 1 "See"

**Pretrained MegaDetector V6 over all 10 PanAf500 clips, on a Colab GPU.**

## How to run this

1. **Runtime → Change runtime type → GPU → Save.** (A100 if you have it; T4 works.) Do this *first* — changing it later restarts
   the session and throws away everything installed.
2. **Runtime → Run all.**
3. Wait ~45 minutes on an A100: the baseline run, then both arms of the variant comparison
   (section 10). No restart is needed — the install cell refreshes the import path itself.
   If Colab prompts to restart anyway, **Restart** then **Runtime → Run all** is safe:
   completed steps are skipped.

Nothing needs editing. No files to upload.

### What it does

Downloads 10 PanAf500 clips (~23 MB) straight from the Bristol deposit, runs MegaDetector over
**every frame of all 10** (3,600 frames), stitches annotated video, and measures accuracy against
the dataset's ground-truth boxes.

### Two things to know

- **The device.** PyTorch-Wildlife accepts `device="cuda"`, stores it, and **never applies it** —
  the weights load on CPU and nothing raises. This notebook forces and *verifies* the placement, so
  watch for `weights forced onto cuda:0` in section 5.
- **Green vs amber.** In the annotated video, **green = MegaDetector prediction**,
  **amber = dataset ground truth** with its behaviour label. MegaDetector only ever outputs
  `animal` — not species, not behaviour.

### Licence

PanAf20K is under a **Non-Commercial Government Licence v2**. Clips downloaded here must not be
redistributed, and the annotated video is a derived work.

## 1. Check the GPU

In [ ]:
import subprocess

out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if out.returncode == 0:
    print([l for l in out.stdout.splitlines() if "MiB" in l or "Tesla" in l or "NVIDIA" in l][:2])
    print("\nGPU OK.")
else:
    raise SystemExit(
        "NO GPU. Runtime > Change runtime type > GPU > Save, then Runtime > Run all."
    )

## 2. Get the code

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/adikothuri3/PanAF-Ape-Detection.git"
REPO_DIR = "/content/PanAF-Ape-Detection"

if not Path(REPO_DIR).exists():
    !git clone --depth 1 -q $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull -q

if not Path(REPO_DIR, "pyproject.toml").is_file():
    raise SystemExit(f"clone failed -- {REPO_DIR} has no pyproject.toml. Check the output above.")

os.chdir(REPO_DIR)
os.environ["PANAF_REPO_ROOT"] = REPO_DIR
print("working in", Path.cwd())

In [ ]:
# Colab's `!command` does NOT stop "Run all" when a command fails -- you get a
# confusing error several cells later instead of the real one. `run()` raises,
# so the notebook halts at the actual failure.
import subprocess
import sys


def run(*args: str) -> None:
    """Run a command, streaming output, and raise if it fails."""
    print("$", " ".join(args), flush=True)
    result = subprocess.run(args, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"`{' '.join(args)}` failed with exit code {result.returncode}. "
            "Read the output above -- that is the real error. Do not continue past this cell."
        )

## 3. Install

**Deliberately does not install `requirements-colab.txt` here.** That file pins the full 170-package
locked environment including `torch`, and forcing it onto Colab replaces the CUDA-matched torch that
is already installed — a multi-gigabyte download that can leave the runtime without working CUDA.

Instead this installs only what Colab lacks and keeps Colab's torch. The locked file remains the
source of truth for reproducing the environment *outside* Colab.

Two or three minutes.

In [ ]:
# Check Python BEFORE pip runs. If Colab's interpreter is outside the range this
# project declares, `pip install -e .` refuses outright with "requires a
# different Python" -- and because the other three packages install fine, the
# only symptom is `panaf_ape_detection` missing several lines later. Catch it
# here, where the message can say what actually happened.
import sys
import tomllib
from pathlib import Path

required = tomllib.loads(Path("pyproject.toml").read_text())["project"]["requires-python"]
running = f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
print(f"Python {running}   pyproject requires {required}")

try:
    from packaging.specifiers import SpecifierSet
    from packaging.version import Version

    supported = Version(f"{sys.version_info.major}.{sys.version_info.minor}") in SpecifierSet(
        required
    )
except ImportError:  # packaging absent -- skip rather than guess
    supported = True
    print("(packaging not installed; skipping the version check)")

if not supported:
    raise SystemExit(
        f"Colab is running Python {running}, which is outside this project's "
        f"declared range {required}. `pip install -e .` will refuse, and the only visible "
        "symptom would be a missing panaf_ape_detection.\n\n"
        "Options: pick a Colab runtime with a supported Python, or widen requires-python in "
        "pyproject.toml if the dependencies genuinely support this version -- do not widen it "
        "blind, the constraint is there because the lockfile was resolved against it."
    )

In [ ]:
# Keep Colab's CUDA-matched torch; add only what is missing.
# setuptools<81 first, because yolov5 (pulled in by PytorchWildlife) still
# imports pkg_resources, which setuptools 81 deprecated and 83 removed.
!pip install -q "setuptools<81"
!pip install -q pytorchwildlife soundfile librosa
!pip install -e . --no-deps   # not -q: if this fails, the reason must be visible

# `pip install -e` drops a .pth file into site-packages, but .pth files are only
# processed at interpreter *startup* -- so in a kernel that is already running,
# the package stays invisible to find_spec even though the install succeeded.
# Subprocesses start fresh and see it fine, which is why the pipeline itself
# works while the check below used to fail. Refresh the path instead of telling
# you to restart.
import importlib
import importlib.util  # `import importlib` alone does not bind .util
import site
import sys
from pathlib import Path

importlib.invalidate_caches()
site.main()  # re-process .pth files, including the one pip just wrote
src = str(Path(REPO_DIR) / "src")  # src layout: belt and braces
if src not in sys.path:
    sys.path.insert(0, src)

# Verify rather than assume -- a failed pip would otherwise surface as a
# confusing error several cells later.
REQUIRED = ("PytorchWildlife", "cv2", "supervision", "panaf_ape_detection")
missing = [m for m in REQUIRED if importlib.util.find_spec(m) is None]
if missing:
    raise SystemExit(
        f"these did not install: {missing}. Read the pip output above -- that is the real "
        "error. If it mentions a version conflict, Runtime > Restart session, then "
        "Runtime > Run all (completed steps are skipped)."
    )
print("install OK:", ", ".join(REQUIRED))

In [ ]:
# Confirm torch still sees the GPU after the install.
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("\nCUDA is missing. Runtime > Restart session, then Runtime > Run all.")

## 4. Verify the stack

`smoke_inference.py` proves the heavy stack actually works — imports, ByteTrack, NumPy interop, a
video round-trip — **without downloading any model weights**. If something is wrong with the
environment, it fails here in seconds rather than twenty minutes into the run.

In [ ]:
run(sys.executable, "scripts/smoke_inference.py")

## 5. Download the clips

Straight from the Bristol deposit — no upload needed. ~23 MB.

Selection is **purposive, not random**: it profiles candidate annotations first (no video), then
greedily picks clips covering all nine behaviours, both species, crowded frames, small and large
subjects, and frames containing no ape. The reason for each pick goes into the manifest.

In [ ]:
run(sys.executable, "scripts/fetch_panaf500.py", "--count", "10", "--pool", "150")

In [ ]:
from pathlib import Path

import pandas as pd

manifest_path = Path("data/sample_manifest.csv")
if not manifest_path.is_file():
    raise SystemExit(
        "No manifest at data/sample_manifest.csv, which means the download cell above did not "
        "finish. Scroll up to it and read its output -- the real error is there. Re-running that "
        "cell is usually enough; network failures to data.bris.ac.uk are typically transient."
    )

manifest = pd.read_csv(manifest_path)
print(f"{len(manifest)} clips selected\n")
for _, row in manifest.iterrows():
    print(f"{row.clip_id}  [{row.split}]  {row.species}")
    print(f"    {row.selected_reason}\n")

## 6. Run detection and tracking on all 10 clips

Every frame of every clip: decode → MegaDetector → confidence filter → **ByteTrack** → compare
against ground truth → draw boxes → stitch to MP4 → write metrics and run metadata.

Tracking is CPU-side association over boxes the detector already produced, so it adds almost nothing
to the runtime. Two tables come out: detection accuracy, and track quality against the dataset's
`ape_id` — ID switches, fragmentation, and how much of each individual was followed.

**~10–15 minutes.** Watch for the device line early on:

```
WARNING ... PyTorch-Wildlife ignored device='cuda' (weights on 'cpu'); forcing it
INFO    ... weights forced onto cuda:0
```

That is the upstream bug being corrected. If it instead says the weights stayed on CPU, stop — the
run would be ~20x slower and the metadata would be wrong.

Clips already finished are skipped, so re-running after a dropped session resumes.

In [ ]:
run(sys.executable, "-m", "panaf_ape_detection.cli", "detect",
    "--config", "configs/colab.yaml")

## 7. Results

Real measurements at the stated confidence and IoU thresholds. A detection counts as correct when it
**localises** an annotated ape — MegaDetector cannot identify species, so no species claim is made.

In [ ]:
import json
from pathlib import Path

import pandas as pd

files = sorted(Path("artifacts/metrics").glob("*.json"))
if not files:
    raise SystemExit(
        "No metrics in artifacts/metrics/, so the detection cell above did not complete. "
        "Scroll up and read its output."
    )

metrics = [json.loads(p.read_text()) for p in files]

table = pd.DataFrame([{
    "clip": m["clip_id"],
    "frames": m["frames_evaluated"],
    "precision": round(m["overall"]["precision"], 3),
    "recall": round(m["overall"]["recall"], 3),
    "f1": round(m["overall"]["f1"], 3),
    "mean_iou": round(m["mean_iou"], 3),
    "empty_frames": m["empty_frames"],
    "FP_on_empty": m["false_positives_on_empty_frames"],
} for m in metrics])
display(table)

tp = sum(m["overall"]["true_positives"] for m in metrics)
fp = sum(m["overall"]["false_positives"] for m in metrics)
fn = sum(m["overall"]["false_negatives"] for m in metrics)
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

print(f"\n{len(metrics)} clips, {sum(m['frames_evaluated'] for m in metrics)} frames")
print(f"TP={tp}  FP={fp}  FN={fn}")
print(f"precision={precision:.3f}  recall={recall:.3f}  F1={f1:.3f}")

### Where it fails

This is the table that should drive any fine-tuning decision. A detector that misses arboreal
postures and small subjects needs different work from one that misses everything equally.

In [ ]:
from collections import defaultdict

behaviour = defaultdict(lambda: [0, 0])
size = defaultdict(lambda: [0, 0])

for m in metrics:
    for label, c in m["by_behaviour"].items():
        behaviour[label][0] += c["true_positives"]
        behaviour[label][1] += c["true_positives"] + c["false_negatives"]
    for band, c in m["by_size"].items():
        size[band][0] += c["true_positives"]
        size[band][1] += c["true_positives"] + c["false_negatives"]

print("Recall by behaviour (worst first)")
for label, (found, total) in sorted(behaviour.items(), key=lambda kv: kv[1][0] / max(kv[1][1], 1)):
    print(f"  {label:20} {found:5}/{total:<6} {found / total:.3f}")

print("\nRecall by subject size (fraction of frame area)")
for band in ("small", "medium", "large"):
    if band in size:
        found, total = size[band]
        print(f"  {band:8} {found:5}/{total:<6} {found / total:.3f}")

## 8. Watch an annotated clip

**Green = MegaDetector prediction** (`#id` from the tracker, then confidence). **Amber = dataset ground truth** (with the
behaviour label and the individual's id). The legend is drawn on every frame, so a still pulled out
of the video is still unambiguous about which box came from where.

In [ ]:
import base64
from pathlib import Path

from IPython.display import HTML, display

videos = sorted(Path("artifacts/videos").glob("*_annotated.mp4"))
print(f"{len(videos)} annotated clips in artifacts/videos/\n")

# Colab's player cannot decode mp4v, so re-encode to H.264 just for display.
for source in videos[:2]:
    playable = source.with_name(source.stem + "_h264.mp4")
    !ffmpeg -y -loglevel error -i "{source}" -vcodec libx264 -pix_fmt yuv420p "{playable}"
    encoded = base64.b64encode(playable.read_bytes()).decode()
    print(source.name)
    display(HTML(
        f'<video width=720 controls><source src="data:video/mp4;base64,{encoded}" '
        f'type="video/mp4"></video>'
    ))

## 9. Keep the outputs

**Colab sessions are ephemeral — anything not copied out is lost.**

Set `USE_DRIVE = True` and re-run this cell to copy `artifacts/` to your Drive. You will be asked to
authorise access.

Do not commit the clips or the annotated video: they are derived works of a non-commercially
licensed dataset, and `artifacts/` and `data/` are git-ignored for that reason.

In [ ]:
USE_DRIVE = False
DESTINATION = "/content/drive/MyDrive/panaf-ape-detection/artifacts"

if USE_DRIVE:
    import shutil

    from google.colab import drive

    drive.mount("/content/drive")
    shutil.copytree("artifacts", DESTINATION, dirs_exist_ok=True)
    print("copied to", DESTINATION)
else:
    print("Drive copy off. Set USE_DRIVE = True and re-run this cell to keep the outputs.")

In [ ]:
# The run-metadata record: commit, config, verified device, variant, threshold,
# seed, input checksums, elapsed time. This is what makes the run reproducible.
import json
from pathlib import Path

for path in sorted(Path("artifacts/metadata").glob("*.json"))[-1:]:
    meta = json.loads(path.read_text())
    for key in ("experiment_name", "git_commit", "git_dirty", "device", "model_variant",
                "confidence_threshold", "seed", "elapsed_seconds"):
        print(f"{key:22} {meta.get(key)}")
    print(f"{'inputs':22} {len(meta.get('inputs', []))} files, checksummed")

## 10. Variant comparison — `MDV6-yolov10-e` (the current open question)

**This is the experiment to run next**, and it is config-only: no code changes, no training.

The 10-clip findings showed detection recall of 0.386 at confidence 0.20, with the misses
concentrated in low-contrast footage where the model still fires but scores **0.05–0.15**. Two
things follow, and this run separates them:

1. Does a larger, higher-resolution variant score those same subjects **higher**?
2. Or is the score depression a property of the footage that no pretrained variant fixes?

Why the *scores* matter and not just recall: `sv.ByteTrack` discards detections at or below **0.1**
and needs `activation + 0.1` to start a track. Recall recovered below 0.15 therefore never reaches
the tracker — measured: dropping the detector to 0.05 changed track coverage by 0.002. A variant
that lifts scores **above the floor** raises coverage; one that merely finds more faint boxes does
not.

Either outcome is decision-grade. Better → adopt it, and the tracker floor stops mattering. Same →
capacity is not the gap, which is the strongest evidence yet that fine-tuning is the real answer.

**Both arms run here**, because a fresh Colab session has no `artifacts/` and the comparison
needs both. Each is detector-only at confidence 0.05, so they are comparable at *every*
threshold rather than only at 0.20 — detections below the threshold are never written, so a run
at 0.20 could not be swept downward afterwards.

Roughly 4× the postprocessing cost of a 0.20 run: budget **~15 minutes per arm on an A100**
(~40 on a T4). Tracking is off in both — `drop_short_tracks` runs before evaluation and would
confound a detector comparison. Run `panaf-phase1 track` over the saved detections afterwards
if you want track quality, without paying for inference twice.

In [ ]:
# BOTH arms, in one session. A fresh Colab has no artifacts/, so the baseline
# must be produced here too -- the comparison cells below read from both.
# ~15 min each on an A100.
run(sys.executable, "-m", "panaf_ape_detection.cli", "detect",       # arm A: yolov9-c
    "--config", "configs/colab-sweep-conf005.yaml", "--overwrite")

run(sys.executable, "-m", "panaf_ape_detection.cli", "detect",       # arm B: yolov10-e
    "--config", "configs/colab-variant-yolov10e.yaml", "--overwrite")

### Compare the two variants

Both runs saved every detection down to 0.05, so this compares them at any threshold without
re-running inference. The last table is the one that decides it: **what fraction of each variant's
detections clear ByteTrack's 0.1 floor.**

In [ ]:
# Both arms saved every detection down to 0.05, so `evaluate --confidence`
# re-scores them at any threshold without touching the GPU again.
ARMS = {
    "yolov9-c  (baseline)": "configs/colab-sweep-conf005.yaml",
    "yolov10-e (new)": "configs/colab-variant-yolov10e.yaml",
}

for label, config in ARMS.items():
    print(f"\n{'=' * 60}\n{label}\n{'=' * 60}")
    for threshold in (0.05, 0.10, 0.20, 0.30):
        run(sys.executable, "-m", "panaf_ape_detection.cli", "evaluate",
            "--config", config, "--confidence", str(threshold))

In [ ]:
# The question that decides adoption: do the new detections clear ByteTrack's
# floor? Recall recovered at or below 0.1 can never start a track, so a variant
# that only finds fainter boxes changes nothing downstream.
import json
from pathlib import Path

ROOTS = {
    "yolov9-c  (baseline)": Path("artifacts/sweep-conf005"),
    "yolov10-e (new)": Path("artifacts/variant-yolov10e"),
}

print(f"{'variant':22} {'total':>7} {'<=0.10':>8} {'0.10-0.15':>10} {'>0.15 usable':>14}")
for label, root in ROOTS.items():
    files = sorted((root / "detections").glob("*.json")) if (root / "detections").is_dir() else []
    if not files:
        print(f"{label:22} not run yet")
        continue
    scores = [d["confidence"]
              for path in files
              for frame in json.loads(path.read_text())["frames"]
              for d in frame["detections"]]
    low = sum(1 for s in scores if s <= 0.10)
    mid = sum(1 for s in scores if 0.10 < s <= 0.15)
    high = len(scores) - low - mid
    print(f"{label:22} {len(scores):7d} {low:8d} {mid:10d} {high:8d} ({high / len(scores):5.0%})")

### Keep the comparison outputs

**Section 9 ran before this comparison existed**, so its Drive copy did not include these results.
This cell saves everything again, after both arms. Colab sessions are ephemeral: the metrics JSONs
are what the write-up has to cite, so losing them means re-running 30 minutes of inference.

The small files are the ones that matter — `metrics/` and `metadata/`. `detections/` is worth
keeping too: it makes every future threshold sweep free.

In [ ]:
SAVE_COMPARISON = True

if SAVE_COMPARISON:
    import shutil

    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    for arm in ("artifacts/sweep-conf005", "artifacts/variant-yolov10e"):
        target = f"/content/drive/MyDrive/panaf-ape-detection/{arm}"
        shutil.copytree(arm, target, dirs_exist_ok=True)
        print("copied", arm, "->", target)
    print("\nBring metrics/ and metadata/ back into the repo to write the result up.")
else:
    print("Not saved. Set SAVE_COMPARISON = True and re-run before closing the session.")

## Next

- Record what you saw in `experiments/experiment_log.md`, **including anything that failed**.
- Bring `metrics/` and `metadata/` from both arms back into the repo, so every number in the
  write-up traces to a file rather than to a screenshot of this notebook.
- **Read the comparison by the score distribution, not the recall row.** A variant that finds more
  faint boxes below 0.1 changes nothing downstream, because `sv.ByteTrack` discards them. A variant
  that lifts the same subjects above 0.15 raises track coverage, which is what caps Phase 2.
- If yolov10-e is no better, that is the result: capacity is not the gap, and the case for
  fine-tuning — or for different footage conditioning — is now evidence-backed rather than assumed.
